# Async LangGraph: Concurrent Tool Calls with asyncio
## A Hands-On Workshop

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/esturban/agent/blob/main/examples/62-async-langgraph/async_langgraph_workbook.ipynb)

**Async LangGraph** lets your agent nodes yield control to the event loop while waiting for I/O — enabling concurrent tool calls, non-blocking streaming, and wall-clock time that doesn't grow linearly with the number of calls. By the end of this workshop you will understand *why* async matters, *how* to write async nodes, and *when* `asyncio.gather()` actually saves time.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Sync vs async; the event loop; why async nodes exist |
| 2 | **Sync Baseline** — Build and time a blocking LangGraph graph |
| 3 | **Async Nodes** — `async def` nodes + `ainvoke()` |
| 4 | **`asyncio.gather()`** — Concurrent tool calls inside a single node |
| 5 | **Streaming** — Token-level output with `astream_events()` |
| 6 | **When to Use Async** — Decision guide + pitfalls |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ with a `.venv` containing the repo's `requirements.txt`, **or** Google Colab
- An `OPENAI_API_KEY` in `.env` (local) or Colab Secrets
- Familiarity with basic LangGraph (`StateGraph`, `add_node`, `add_edge`) — see example 01

### Key References
> Python Software Foundation. *asyncio — Asynchronous I/O.* https://docs.python.org/3/library/asyncio.html
> LangGraph team. *How to run graph asynchronously.* https://langchain-ai.github.io/langgraph/how-tos/async/
> LangGraph team. *How to stream events from your graph.* https://langchain-ai.github.io/langgraph/how-tos/streaming-events-from-your-graph/
> Hattingh, C. (2020). *Using Asyncio in Python.* O'Reilly. — Chapter 4: Asyncio in the library ecosystem.

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain",
            "langchain-openai",
            "langgraph",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon), name = OPENAI_API_KEY
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.getenv("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY not found or looks wrong. "
    "Local: echo 'OPENAI_API_KEY=sk-...' >> .env  "
    "Colab: Secrets panel → Add secret 'OPENAI_API_KEY'"
)
print(f"API key loaded: {key[:8]}...")

In [ ]:
import asyncio
import time
from typing import TypedDict

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Imports ready. LLM initialised.")

---

## Part 1 — Concepts: Sync vs Async

---

### The Problem with Blocking Code

A standard LangGraph node calls `llm.invoke()`, which **blocks the thread** until the API responds — usually 500 ms–3 s per call. If a node needs three pieces of data (weather, news, calendar), each call blocks the next:

```
SYNC — sequential (wall-clock ~3 s for 3 × 1 s tools)
────────────────────────────────────────────────────────
Thread:  [== fetch_weather ==][== fetch_news ==][== fetch_calendar ==]
         0s                   1s               2s                   3s
```

With `async/await` and `asyncio.gather()`, all three fire simultaneously — the event loop switches between them while each waits for its I/O response:

```
ASYNC — concurrent (wall-clock ~1 s for 3 × 1 s tools)
────────────────────────────────────────────────────────
Event loop:
  fetch_weather  [== waiting for I/O ==]
  fetch_news     [== waiting for I/O ==]
  fetch_calendar [== waiting for I/O ==]
                 0s                     1s
                 ^ all started at once  ^ all done at once
```

**Speedup = number of concurrent I/O calls** (for I/O-bound work).

---

### Key Vocabulary

| Term | Definition |
|------|------------|
| **coroutine** | A function defined with `async def`; suspendable at `await` points |
| **event loop** | The asyncio scheduler that runs coroutines cooperatively |
| **`await`** | Suspend this coroutine and give control back to the event loop |
| **`asyncio.gather()`** | Run multiple coroutines concurrently; collect all results |
| **`ainvoke()`** | Async version of LangChain/LangGraph `.invoke()` |
| **`astream_events()`** | Async generator that yields token-level events as they arrive |
| **I/O-bound** | Time dominated by waiting (network, disk) — async helps here |
| **CPU-bound** | Time dominated by computation — async does NOT help; use multiprocessing |

---

### Sync vs Async LangGraph at a Glance

| | Sync | Async |
|---|---|---|
| Node signature | `def node(state)` | `async def node(state)` |
| LLM call | `llm.invoke(...)` | `await llm.ainvoke(...)` |
| Run graph | `app.invoke(...)` | `await app.ainvoke(...)` |
| Concurrent calls | Not possible in one node | `await asyncio.gather(...)` |
| Token streaming | `.stream(...)` | `async for ... in app.astream_events(...)` |
| Jupyter support | Works everywhere | Works in Jupyter (event loop already running) |
| Best for | Simple scripts, CPU-bound | APIs, multiple I/O per node, streaming UIs |

---

### `asyncio.gather()` Patterns

```python
# Pattern 1 — fixed list of coroutines
a, b, c = await asyncio.gather(coro_a(), coro_b(), coro_c())

# Pattern 2 — dynamic list (unpack with *)
tasks = [fetch(url) for url in url_list]
results = await asyncio.gather(*tasks)

# Pattern 3 — gather with error handling (return_exceptions=True)
results = await asyncio.gather(*tasks, return_exceptions=True)
for r in results:
    if isinstance(r, Exception):
        print(f"Task failed: {r}")
    else:
        print(f"Got: {r}")
```

> **Rule of thumb:** If a node makes **more than one external call**, `asyncio.gather()` is almost always worth it.

---

## Part 2 — Sync Baseline

---

Before switching to async, let's build a standard sync graph and measure how long it takes. This gives us the baseline we'll compare against in Parts 3 and 4.

The graph has two nodes that each call the LLM once — sequentially, one after the other. This is how most LangGraph tutorials start: simple, familiar, and measurably slow when you need multiple calls.

In [ ]:
# ─── 2-A: Define a sync two-node graph ────────────────────────────────────────
# Each node blocks while the LLM responds.
# Total time ≈ llm_call_1 + llm_call_2 (sequential).


class SyncState(TypedDict):
    question: str
    short_answer: str
    emoji_summary: str


def sync_answer_node(state: SyncState) -> dict:
    """Node 1 — answer the question (blocks until LLM responds)."""
    response = llm.invoke([HumanMessage(content=state["question"])])
    return {"short_answer": response.content}


def sync_emoji_node(state: SyncState) -> dict:
    """Node 2 — summarise the answer as a single emoji (also blocks)."""
    prompt = f"Reply with exactly one emoji that represents: {state['short_answer']}"
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"emoji_summary": response.content.strip()}


sync_builder = StateGraph(SyncState)
sync_builder.add_node("answer", sync_answer_node)
sync_builder.add_node("emoji", sync_emoji_node)
sync_builder.add_edge(START, "answer")
sync_builder.add_edge("answer", "emoji")
sync_builder.add_edge("emoji", END)
sync_app = sync_builder.compile()

print("Sync graph compiled — two nodes, run sequentially.")

In [ ]:
# ─── 2-B: Time the sync graph ─────────────────────────────────────────────────
# Record wall-clock time for a single invocation.
# Keep this number — we will compare against async in Part 3.

t0 = time.perf_counter()
sync_result = sync_app.invoke(
    {"question": "Name one planet in the solar system.", "short_answer": "", "emoji_summary": ""}
)
sync_elapsed = time.perf_counter() - t0

print(f"Answer  : {sync_result['short_answer'].strip()[:80]}")
print(f"Emoji   : {sync_result['emoji_summary']}")
print(f"Time    : {sync_elapsed:.2f}s  (two sequential LLM calls)")
print()
print("Keep this time in mind — we will compare it against async in Part 3.")

---

## Part 3 — Async Nodes + `ainvoke()`

---

Switching to `async def` and `await llm.ainvoke()` costs almost nothing for a **single sequential call** — latency is about the same. The difference shows up when we run calls concurrently (Part 4). But using async nodes from the start is a good habit: it keeps the door open for `gather()` later and is required for `astream_events()`.

### What changes when you go async

```python
# BEFORE — sync node
def my_node(state: MyState) -> dict:
    response = llm.invoke([HumanMessage(content=state["q"])])
    return {"answer": response.content}

# AFTER — async node (two changes: async def + await)
async def my_node(state: MyState) -> dict:
    response = await llm.ainvoke([HumanMessage(content=state["q"])])
    return {"answer": response.content}
```

The graph definition (`StateGraph`, `add_node`, `add_edge`, `compile`) is **identical** — you only change the node functions and the call site (`ainvoke` instead of `invoke`).

In [ ]:
# ─── 3-A: Convert nodes to async ──────────────────────────────────────────────
# Only two changes per node: `async def` and `await llm.ainvoke()`.
# The StateGraph wiring is identical to the sync version.


class AsyncState(TypedDict):
    question: str
    short_answer: str
    emoji_summary: str


async def async_answer_node(state: AsyncState) -> dict:
    """Node 1 — async. Yields to event loop while waiting for I/O."""
    response = await llm.ainvoke([HumanMessage(content=state["question"])])
    return {"short_answer": response.content}


async def async_emoji_node(state: AsyncState) -> dict:
    """Node 2 — async."""
    prompt = f"Reply with exactly one emoji that represents: {state['short_answer']}"
    response = await llm.ainvoke([HumanMessage(content=prompt)])
    return {"emoji_summary": response.content.strip()}


async_builder = StateGraph(AsyncState)
async_builder.add_node("answer", async_answer_node)
async_builder.add_node("emoji", async_emoji_node)
async_builder.add_edge(START, "answer")
async_builder.add_edge("answer", "emoji")
async_builder.add_edge("emoji", END)
async_app = async_builder.compile()

print("Async graph compiled — same structure, same wiring, async nodes.")

In [ ]:
# ─── 3-B: Run with ainvoke() and compare ──────────────────────────────────────
# In Jupyter the event loop is already running, so we can `await` directly.
# In a plain script you'd use: asyncio.run(async_app.ainvoke(...))

t0 = time.perf_counter()
async_result = await async_app.ainvoke(
    {"question": "Name one planet in the solar system.", "short_answer": "", "emoji_summary": ""}
)
async_elapsed = time.perf_counter() - t0

print(f"Answer  : {async_result['short_answer'].strip()[:80]}")
print(f"Emoji   : {async_result['emoji_summary']}")
print(f"Time    : {async_elapsed:.2f}s  (two sequential async LLM calls)")
print()
print(f"Sync baseline : {sync_elapsed:.2f}s")
print(f"Async (seq)   : {async_elapsed:.2f}s")
print()
print("Result: similar latency — the gain appears when calls run CONCURRENTLY (Part 4).")

---

## Part 4 — `asyncio.gather()` for Concurrent Tool Calls

---

This is where async pays off. Inside one node, fire **multiple I/O-bound calls simultaneously** using `asyncio.gather()`. While one tool waits for its response, the others are already in flight.

```
CONCURRENT TOOL CALLS inside one LangGraph node
────────────────────────────────────────────────────────────────────

  gather_context_node
       |
       +--- asyncio.gather() ----------------------------------------
       |         |                  |                  |
       |    [fetch_weather]    [fetch_news]    [fetch_calendar]
       |    await sleep(1s)    await sleep(1s)  await sleep(1s)
       |         |                  |                  |
       |         +------------------+------------------+
       |                       ~1 s total
       |
       v
  context = "Sunny, 22C | Markets up | 3 meetings"
       |
  summarise_node  ->  await llm.ainvoke()
       |
       v
  "Good morning! Clear skies, markets positive, busy day ahead."
```

**Sequential time:** 3 x 1 s = 3 s
**Concurrent time:** max(1 s, 1 s, 1 s) = ~1 s
**Speedup:** proportional to the number of concurrent I/O calls

In [ ]:
# ─── 4-A: Define simulated I/O-bound tools ────────────────────────────────────
# Each sleeps for 1 second — simulating a real API call.
# In production these would be: fetch_weather(), search_news(), query_calendar(), etc.


async def fetch_weather() -> str:
    await asyncio.sleep(1)
    return "Sunny, 22 C"


async def fetch_news() -> str:
    await asyncio.sleep(1)
    return "Markets up 0.3%"


async def fetch_calendar() -> str:
    await asyncio.sleep(1)
    return "3 meetings today"


# ── Sequential: await one at a time ─────────────────────────────────────────
async def sequential_gather(state: dict) -> dict:
    weather = await fetch_weather()
    news = await fetch_news()
    calendar = await fetch_calendar()
    return {"context": f"{weather} | {news} | {calendar}"}


# ── Concurrent: all at once with asyncio.gather() ────────────────────────────
async def concurrent_gather(state: dict) -> dict:
    weather, news, calendar = await asyncio.gather(
        fetch_weather(),
        fetch_news(),
        fetch_calendar(),
    )
    return {"context": f"{weather} | {news} | {calendar}"}


# Time both
t0 = time.perf_counter()
await sequential_gather({})
seq_time = time.perf_counter() - t0

t0 = time.perf_counter()
await concurrent_gather({})
con_time = time.perf_counter() - t0

print(f"Sequential  : {seq_time:.2f}s  (3 tools, one at a time)")
print(f"Concurrent  : {con_time:.2f}s  (3 tools, asyncio.gather)")
print(f"Speedup     : {seq_time / con_time:.1f}x")

In [ ]:
# ─── 4-B: Wire gather_context into a real LangGraph ───────────────────────────
# Graph: gather_context -> summarise
# The gather node fires three tools concurrently; summarise calls the LLM once.


class BriefingState(TypedDict):
    context: str
    summary: str


async def gather_context_node(state: BriefingState) -> dict:
    weather, news, calendar = await asyncio.gather(
        fetch_weather(),
        fetch_news(),
        fetch_calendar(),
    )
    return {"context": f"Weather: {weather}. News: {news}. Calendar: {calendar}."}


async def summarise_node(state: BriefingState) -> dict:
    prompt = f"Give a one-sentence morning briefing from: {state['context']}"
    response = await llm.ainvoke([HumanMessage(content=prompt)])
    return {"summary": response.content}


briefing_builder = StateGraph(BriefingState)
briefing_builder.add_node("gather", gather_context_node)
briefing_builder.add_node("summarise", summarise_node)
briefing_builder.add_edge(START, "gather")
briefing_builder.add_edge("gather", "summarise")
briefing_builder.add_edge("summarise", END)
briefing_app = briefing_builder.compile()

t0 = time.perf_counter()
out = await briefing_app.ainvoke({"context": "", "summary": ""})
elapsed = time.perf_counter() - t0

print(f"Context : {out['context']}")
print(f"Summary : {out['summary']}")
print(f"Time    : {elapsed:.2f}s  (3 concurrent tools + 1 LLM call)")

---

## Part 5 — Streaming with `astream_events()`

---

`astream_events()` is an async generator that yields events as the graph executes — including token-by-token LLM output. This powers streaming UIs where the user sees text appear as it's generated rather than waiting for the full response.

### Event types

| Event | When it fires | Useful data |
|-------|--------------|-------------|
| `on_chain_start` | Graph or node begins | `event["name"]` |
| `on_chain_end` | Graph or node finishes | `event["data"]["output"]` |
| `on_chat_model_start` | LLM call begins | model name |
| `on_chat_model_stream` | Each token arrives | `event["data"]["chunk"].content` |
| `on_chat_model_end` | LLM call finishes | full response |

### `astream()` vs `astream_events()`

| | `astream()` | `astream_events()` |
|---|---|---|
| What it yields | Full node output dicts after each node completes | Fine-grained events including individual tokens |
| Token streaming | No | Yes (`on_chat_model_stream`) |
| Use for | Watching graph progress node by node | Live token-by-token UI output |
| API | `async for chunk in app.astream(input)` | `async for event in app.astream_events(input, version="v2")` |

In [ ]:
# ─── 5-A: astream() — node-level progress ────────────────────────────────────
# Each chunk is the output dict of a completed node.
# Good for progress indicators ("node X done") but no token streaming.

print("Streaming node outputs with astream():")
print()
async for chunk in briefing_app.astream({"context": "", "summary": ""}):
    for node_name, node_output in chunk.items():
        print(f"  [{node_name}] -> {str(node_output)[:100]}")

In [ ]:
# ─── 5-B: astream_events() — token-level streaming ───────────────────────────
# Filter on on_chat_model_stream to print each token as it arrives.
# This is what powers live-typing UIs.


class StreamState(TypedDict):
    question: str
    answer: str


async def stream_node(state: StreamState) -> dict:
    response = await llm.ainvoke([HumanMessage(content=state["question"])])
    return {"answer": response.content}


stream_builder = StateGraph(StreamState)
stream_builder.add_node("agent", stream_node)
stream_builder.add_edge(START, "agent")
stream_builder.add_edge("agent", END)
stream_app = stream_builder.compile()

print("Streaming tokens (on_chat_model_stream events):")
print()
print("Response: ", end="", flush=True)
async for event in stream_app.astream_events(
    {"question": "List 3 programming languages, one per line.", "answer": ""},
    version="v2",
):
    if event["event"] == "on_chat_model_stream":
        chunk = event["data"]["chunk"].content
        if chunk:
            print(chunk, end="", flush=True)
print()
print("[stream complete]")

In [ ]:
# ─── 5-C: Inspect all event types ─────────────────────────────────────────────
# Run a small graph and tally every event type — useful for debugging.

print("All events from a single-node graph run:")
print()
event_counts: dict = {}

async for event in stream_app.astream_events(
    {"question": "Say hi in one word.", "answer": ""},
    version="v2",
):
    etype = event["event"]
    event_counts[etype] = event_counts.get(etype, 0) + 1

print(f"{'Event type':<35} {'Count':>6}")
print("-" * 43)
for etype, count in sorted(event_counts.items()):
    print(f"{etype:<35} {count:>6}")

---

## Part 6 — When to Use Async: Decision Guide

---

### Async saves time only for I/O-bound work

```
Is your node bottlenecked by WAITING (network / disk)?
        |
       YES                              NO
        |                               |
Does it make MORE THAN ONE          Use sync node.
external call per node?             Async adds no value
        |                           for CPU-bound work.
       YES              NO
        |                |
Use asyncio.gather()   Use async def + ainvoke().
(concurrent)           Single-call async is fine --
Speedup = N calls.     prepares you for gather later.
```

---

### Common Pitfalls

| Pitfall | What happens | Fix |
|---------|-------------|-----|
| Mixing `invoke()` inside `async def` | Blocks the event loop during the call | Replace with `await ainvoke()` |
| Using `asyncio.run()` in Jupyter | `RuntimeError: event loop already running` | Just use `await` directly in Jupyter cells |
| Using gather for CPU-bound work | No speedup (GIL still applies) | Use `concurrent.futures.ProcessPoolExecutor` |
| Not handling exceptions in gather | One failure silently suppresses others | Pass `return_exceptions=True` |
| Forgetting `await` on a coroutine | Coroutine object created but never run | Always `await` or gather coroutines |

---

### `asyncio.gather()` vs `asyncio.create_task()` vs sequential

| Approach | Wall-clock | Order of results | Error handling |
|----------|-----------|-----------------|----------------|
| Sequential `await` | Slowest (sum of all) | Predictable | Exception stops execution |
| `asyncio.gather()` | Fastest (max of all) | Same order as input | All must succeed by default |
| `asyncio.create_task()` | Same as gather | Non-deterministic | Must collect with `await task` |

> **Recommendation:** Use `asyncio.gather()` for the vast majority of concurrent tool patterns in LangGraph nodes.

---

## Exercises

---

### Exercise 1 — Convert a Sync Graph to Async

The cell below defines a sync graph that translates a phrase to Spanish and French with **two sequential LLM calls**. Rewrite it so both translations happen concurrently using `asyncio.gather()` inside an async node.

**Goal:** Time both versions. The async concurrent version should be roughly 2x faster.

**Hint:** `llm.ainvoke()` is the async equivalent of `llm.invoke()`.

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
# Sync baseline — two sequential LLM calls.


class TranslateState(TypedDict):
    text: str
    spanish: str
    french: str


def sync_translate_node(state: TranslateState) -> dict:
    es = llm.invoke([HumanMessage(content=f"Translate to Spanish only: {state['text']}")])
    fr = llm.invoke([HumanMessage(content=f"Translate to French only: {state['text']}")])
    return {"spanish": es.content, "french": fr.content}


sync_translate_builder = StateGraph(TranslateState)
sync_translate_builder.add_node("translate", sync_translate_node)
sync_translate_builder.add_edge(START, "translate")
sync_translate_builder.add_edge("translate", END)
sync_translate_app = sync_translate_builder.compile()

t0 = time.perf_counter()
sync_out = sync_translate_app.invoke({"text": "Good morning", "spanish": "", "french": ""})
sync_translate_time = time.perf_counter() - t0
print(f"Sync   ({sync_translate_time:.2f}s): ES={sync_out['spanish']}  FR={sync_out['french']}")

# TODO: rewrite sync_translate_node as an async node using asyncio.gather()
# TODO: build the async graph and run with ainvoke()
# TODO: compare wall-clock time with the sync version above

### Exercise 2 — Add a Fourth Concurrent Tool

Add a `fetch_stock()` async tool (1-second delay, returns `"AAPL +1.2%"`) to the `gather_context_node` from Part 4. Rebuild the briefing graph and verify the total time stays around 1 second even with four tools.

**Bonus:** Add `return_exceptions=True` to `asyncio.gather()` and simulate one tool failing by raising an exception — print a warning for failed tools and continue with the rest.

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────


async def fetch_stock() -> str:
    await asyncio.sleep(1)
    return "AAPL +1.2%"


# TODO: add fetch_stock() to asyncio.gather() in a new gather_context_4 node
# TODO: rebuild the BriefingState graph with the new node
# TODO: time the run — confirm total stays ~1 s not ~4 s

# Bonus: simulate a failed tool
# async def fetch_stock_failing() -> str:
#     await asyncio.sleep(0.5)
#     raise RuntimeError("Stock API down")
#
# results = await asyncio.gather(fetch_weather(), fetch_stock_failing(), return_exceptions=True)
# for name, result in zip(["weather", "stock"], results):
#     if isinstance(result, Exception):
#         print(f"  WARNING: {name} failed: {result}")
#     else:
#         print(f"  {name}: {result}")

---

## What's Next?

You now have a working foundation in async LangGraph. Here's where to go deeper:

### Explore parallel at the graph level
- **Example 28 — Parallel Subgraph** (`../28-parallel-subgraph/`) — fan-out at the graph topology level rather than inside a single node. Use when different branches are independent agents, not just tool calls.

### Handle errors gracefully
- Pass `return_exceptions=True` to `asyncio.gather()` so one failing tool doesn't abort the whole node.
- Use `asyncio.wait_for(coro, timeout=5.0)` to add per-tool timeouts — critical for production.

### Observe async graphs in production
- **LangSmith** traces async graphs identically to sync ones — every node, every tool call, latency histogram.
- Enable with `LANGCHAIN_TRACING_V2=true` + `LANGCHAIN_API_KEY=...` in your `.env` — no code changes needed.

### Batch many inputs
- **Example 72 — Batch Agent Runner** (`../72-batch-agent-runner/`) — run the same graph over hundreds of inputs with asyncio concurrency and retry logic.

### Further reading
- Python asyncio official docs: https://docs.python.org/3/library/asyncio.html
- LangGraph async how-to: https://langchain-ai.github.io/langgraph/how-tos/async/
- LangGraph streaming events: https://langchain-ai.github.io/langgraph/how-tos/streaming-events-from-your-graph/
- Hattingh, C. (2020). *Using Asyncio in Python.* O'Reilly. — Chapter 4: Asyncio in the library ecosystem.

---

*Workshop complete. The next step is example 28 — fan-out at the graph level.*

---

## Exercise Answer Key

Use this section as a reference **after** attempting the exercises yourself. These are sample solutions — not the only correct answers.

---

### Exercise 1 — Convert a Sync Graph to Async

**What to change:**
1. `def` -> `async def` on the node function
2. Two separate `llm.invoke()` calls -> `await asyncio.gather(llm.ainvoke(...), llm.ainvoke(...))`
3. `app.invoke(...)` -> `await app.ainvoke(...)`

**Expected speedup:** approximately 2x (two concurrent LLM calls instead of sequential).

### Exercise 2 — Four Concurrent Tools

**What to change:** Add `fetch_stock()` to the `asyncio.gather()` call. The total time is still `max(1s, 1s, 1s, 1s)` = ~1 s because all four tools run in parallel.

**Bonus:** `return_exceptions=True` prevents one failing tool from raising and killing the entire `gather()` call — essential for production resilience.

In [ ]:
# Answer 1 — concurrent translation node


async def async_translate_node(state: TranslateState) -> dict:
    es_msg = HumanMessage(content=f"Translate to Spanish only: {state['text']}")
    fr_msg = HumanMessage(content=f"Translate to French only: {state['text']}")
    # Both LLM calls fire simultaneously — total time ~ max(es_time, fr_time)
    es, fr = await asyncio.gather(
        llm.ainvoke([es_msg]),
        llm.ainvoke([fr_msg]),
    )
    return {"spanish": es.content, "french": fr.content}


async_translate_builder = StateGraph(TranslateState)
async_translate_builder.add_node("translate", async_translate_node)
async_translate_builder.add_edge(START, "translate")
async_translate_builder.add_edge("translate", END)
async_translate_app = async_translate_builder.compile()

t0 = time.perf_counter()
async_out = await async_translate_app.ainvoke({"text": "Good morning", "spanish": "", "french": ""})
async_translate_time = time.perf_counter() - t0

print(f"Sync     ({sync_translate_time:.2f}s): ES={sync_out['spanish']}  FR={sync_out['french']}")
print(f"Async    ({async_translate_time:.2f}s): ES={async_out['spanish']}  FR={async_out['french']}")
print(f"Speedup  : {sync_translate_time / async_translate_time:.1f}x")

In [ ]:
# Answer 2 — four concurrent tools, still ~1 s


async def gather_context_4(state: BriefingState) -> dict:
    weather, news, calendar, stock = await asyncio.gather(
        fetch_weather(),
        fetch_news(),
        fetch_calendar(),
        fetch_stock(),  # <- added
    )
    return {
        "context": f"Weather: {weather}. News: {news}. Calendar: {calendar}. Stock: {stock}."
    }


b4_builder = StateGraph(BriefingState)
b4_builder.add_node("gather", gather_context_4)
b4_builder.add_node("summarise", summarise_node)
b4_builder.add_edge(START, "gather")
b4_builder.add_edge("gather", "summarise")
b4_builder.add_edge("summarise", END)
b4_app = b4_builder.compile()

t0 = time.perf_counter()
out4 = await b4_app.ainvoke({"context": "", "summary": ""})
elapsed4 = time.perf_counter() - t0

print(f"Context  : {out4['context']}")
print(f"Summary  : {out4['summary']}")
print(f"Time     : {elapsed4:.2f}s  (4 tools, still ~1 s — all concurrent)")
print()
print("Bonus — error handling with return_exceptions=True:")


async def fetch_stock_failing() -> str:
    await asyncio.sleep(0.5)
    raise RuntimeError("Stock API timeout")


results = await asyncio.gather(
    fetch_weather(),
    fetch_news(),
    fetch_stock_failing(),
    return_exceptions=True,
)
for name, result in zip(["weather", "news", "stock"], results):
    if isinstance(result, Exception):
        print(f"  WARNING: {name} failed — {result}")
    else:
        print(f"  {name}: {result}")